# Analisis Regresi Daya Beli: Pengaruh Inflasi dan Variabel Ekonomi Makro
Notebook ini disusun untuk memodelkan dan menginterpretasikan pengaruh berbagai indikator ekonomi makro, khususnya inflasi, terhadap daya beli masyarakat (diwakili oleh Pengeluaran per Kapita). 

Berdasarkan tinjauan statistik, data makroekonomi seringkali memiliki dua tantangan utama:
1. **Missing Value yang bersifat non-random/struktural**: Variabel seperti TPAK dan Persentase Penduduk Miskin memiliki banyak kekosongan, sehingga tidak tepat untuk diimputasi.
2. **Multikolinearitas yang Tinggi**: Indikator seperti UMP dan PDRB memiliki korelasi kuat satu sama lain. Memasukkan semua variabel ke dalam satu model regresi standar akan mengacaukan interpretasi koefisien.

Oleh karena itu, strategi pemodelan dibagi menjadi dua jalur utama:
* **Model Interpretatif (OLS/Regresi Linear)**: Menggunakan subset variabel yang tidak redundan untuk menjelaskan *arah dan signifikansi* pengaruh inflasi terhadap daya beli.
* **Model Prediktif (Ridge Regression)**: Menggunakan regularisasi L2 untuk memberikan prediksi akurat dan ekstrapolasi tren harga di masa depan dengan mengakomodasi lebih banyak fitur.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import scipy.stats as stats
import joblib
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 1. Load Data & Pembersihan Awal
Kita memuat data dari `datasets/processed/clean_daya_beli.csv`. Sesuai kaidah, baris atau kolom dengan *missing values* struktural dieliminasi tanpa imputasi.

In [ ]:
data_path = os.path.join('..', 'datasets', 'processed', 'clean_daya_beli.csv')
df = pd.read_csv(data_path)

# Drop variabel dengan missing values tinggi secara struktural
df_clean = df.drop(columns=['TPAK', 'Pct_Penduduk_Miskin'])

# Pisahkan komponen target agar tidak terjadi leakage
df_clean = df_clean.drop(columns=['Pengeluaran_Makanan', 'Pengeluaran_Bukan_Makanan'])

print("Shape setelah drop missing value kolom:", df_clean.shape)
df_clean.head()

## 2. Feature Engineering & Eliminasi Redundansi
Untuk mendapatkan analisis yang bersih, kita melakukan penyesuaian:
- Menghitung **Real UMP**: UMP yang disesuaikan dengan inflasi.
- Mengeliminasi fitur redundan: Memilih `Real UMP` ketimbang `UMP` nominal, dan memilih `PDRB Harga Konstan` ketimbang `PDRB Harga Berlaku`.

In [ ]:
# Feature Engineering
df_feat = df_clean.copy()
df_feat['Real_UMP'] = df_feat['UMP'] / (1 + df_feat['Inflasi_Rata_Tahunan'])

# Hapus fitur redundan untuk mencegah multikolinearitas ekstrim
df_model = df_feat.drop(columns=['UMP', 'PDRB_HargaBerlaku'])

print("Kolom yang siap digunakan:", df_model.columns.tolist())
df_model.head()

## 3. Train-Test Split (Chronological)
Pelatihan model dilakukan menggunakan data historis (2021-2023), sedangkan pengujian (*testing*) dilakukan pada data masa depan (2024-2025) untuk menguji seberapa baik model mengekstrapolasi tren ekonomi.

In [ ]:
train_mask = df_model['Tahun'] <= 2023
test_mask = df_model['Tahun'] >= 2024

train_df = df_model[train_mask]
test_df = df_model[test_mask]

target_col = 'Total_Pengeluaran'
print(f"Data Training (2021-2023): {train_df.shape[0]} observasi")
print(f"Data Testing (2024-2025): {test_df.shape[0]} observasi")

# Helper function untuk metrik evaluasi
def print_metrics(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- Evaluasi {model_name} ---")
    print(f"MAE  : Rp {mae:,.2f}")
    print(f"RMSE : Rp {rmse:,.2f}")
    print(f"R-sq : {r2:.4f}\n")

## 4. Skenario 1: Model Interpretatif Dasar
**Tujuan**: Melihat murni pengaruh Inflasi dan Real UMP terhadap Pengeluaran per Kapita seiring berjalannya waktu.
**Variabel Independen**: `Tahun`, `Inflasi_Rata_Tahunan`, `Real_UMP`.

In [ ]:
features_s1 = ['Tahun', 'Inflasi_Rata_Tahunan', 'Real_UMP']

X_train_s1 = sm.add_constant(train_df[features_s1])
y_train_s1 = train_df[target_col]

# Uji VIF (Variance Inflation Factor)
vif_data_s1 = pd.DataFrame()
vif_data_s1["Variabel"] = X_train_s1.columns
vif_data_s1["VIF"] = [variance_inflation_factor(X_train_s1.values, i) for i in range(X_train_s1.shape[1])]
print("--- Uji VIF Skenario 1 ---")
print(vif_data_s1[vif_data_s1['Variabel'] != 'const'])
print("\n(VIF < 5 menandakan tidak ada masalah multikolinearitas)\n")

# Model Regresi OLS
ols_s1 = sm.OLS(y_train_s1, X_train_s1).fit()
print(ols_s1.summary())

**Interpretasi Skenario 1:**
- Nilai VIF untuk variabel Inflasi dan Real UMP berada di angka yang sangat rendah (~1.x), membuktikan bahwa model ini **bebas dari masalah multikolinearitas**.
- Kita dapat menginterpretasikan koefisien secara langsung (ceteris paribus). Perhatikan nilai `P>|t|` untuk melihat signifikansi secara statistik.

## 5. Skenario 2: Model Interpretatif Lengkap
**Tujuan**: Memasukkan lebih banyak konteks ekonomi riil.
**Variabel Independen**: Skenario 1 + `PDRB_HargaKonstan` + `TPT` (Tingkat Pengangguran Terbuka).

In [ ]:
features_s2 = ['Tahun', 'Inflasi_Rata_Tahunan', 'Real_UMP', 'PDRB_HargaKonstan', 'TPT']

X_train_s2 = sm.add_constant(train_df[features_s2])
y_train_s2 = train_df[target_col]

# Uji VIF
vif_data_s2 = pd.DataFrame()
vif_data_s2["Variabel"] = X_train_s2.columns
vif_data_s2["VIF"] = [variance_inflation_factor(X_train_s2.values, i) for i in range(X_train_s2.shape[1])]
print("--- Uji VIF Skenario 2 ---")
print(vif_data_s2[vif_data_s2['Variabel'] != 'const'])
print("\n")

# Model Regresi OLS
ols_s2 = sm.OLS(y_train_s2, X_train_s2).fit()
print(ols_s2.summary())

**Interpretasi Skenario 2:**
Dengan penambahan PDRB dan TPT, koefisien beberapa variabel bisa berubah atau level signifikansinya menurun. Jika terjadi peningkatan VIF yang cukup besar, hal tersebut merupakan gejala wajar pada data ekonomi makro wilayah, namun selama R-squared naik dan arah koefisien masuk akal secara ekonomi, model ini berguna untuk analisis komprehensif.

## 6. Skenario 3: Model Prediktif Final (Ridge Regression)
**Tujuan**: Memaksimalkan akurasi prediksi untuk Total Pengeluaran dengan menggunakan seluruh variabel yang relevan, termasuk nama `Provinsi`. 
Karena kita memasukkan banyak variabel (termasuk fitur one-hot encoding provinsi), kita menggunakan **Ridge Regression** (L2 Regularization) untuk menekan *overfitting* dan menangani korelasi antar-fitur agar model mampu melakukan ekstrapolasi waktu secara stabil.

In [ ]:
features_s3_num = ['Tahun', 'Inflasi_Rata_Tahunan', 'Real_UMP', 'PDRB_HargaKonstan', 'TPT']
features_s3_cat = ['Provinsi']

X_train_s3 = train_df[features_s3_num + features_s3_cat]
y_train_s3 = train_df[target_col]

X_test_s3 = test_df[features_s3_num + features_s3_cat]
y_test_s3 = test_df[target_col]

# Pipeline Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_s3_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_s3_cat)
    ]
)

# Pipeline Model Ridge
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0))
])

ridge_pipeline.fit(X_train_s3, y_train_s3)

y_pred_train_ridge = ridge_pipeline.predict(X_train_s3)
y_pred_test_ridge = ridge_pipeline.predict(X_test_s3)

print_metrics("Ridge Train", y_train_s3, y_pred_train_ridge)
print_metrics("Ridge Test", y_test_s3, y_pred_test_ridge)

### 6.1. Feature Importance (Standardized Coefficients)
Dalam Ridge, kita melihat pentingnya fitur dari besaran absolut koefisiennya setelah standarisasi.

In [ ]:
# Mengambil nama fitur setelah One-Hot Encoding
cat_encoder = ridge_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(features_s3_cat)
all_feature_names = features_s3_num + list(cat_feature_names)

# Mengambil koefisien Ridge
ridge_coefs = ridge_pipeline.named_steps['regressor'].coef_

feat_imp = pd.DataFrame({'Feature': all_feature_names, 'Coefficient': ridge_coefs})
feat_imp['Abs_Coef'] = np.abs(feat_imp['Coefficient'])
feat_imp = feat_imp.sort_values(by='Abs_Coef', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Coefficient', y='Feature', data=feat_imp.head(15), palette='coolwarm')
plt.title('Top 15 Feature Importance (Ridge Regression Coefficients)')
plt.xlabel('Standardized Coefficient')
plt.ylabel('Features')
plt.show()

## 7. Uji Asumsi Klasik Residual (Model Prediktif)
Agar model dapat dipercaya keandalannya, kita memastikan bahwa *error* (residual) bersifat acak dan normal.

In [ ]:
residuals = y_test_s3 - y_pred_test_ridge

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Normalitas Residual (Histogram)
sns.histplot(residuals, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribusi Residual (Normalitas)')
axes[0].set_xlabel('Residual (Error)')

# 2. Uji Heteroskedastisitas (Scatter plot Residual vs Prediksi)
sns.scatterplot(x=y_pred_test_ridge, y=residuals, ax=axes[1], color='red', alpha=0.6)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_title('Residual vs Prediksi (Heteroskedastisitas)')
axes[1].set_xlabel('Nilai Prediksi Daya Beli')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

**Interpretasi Grafik:**
- **Distribusi Residual**: Jika kurva berbentuk menyerupai lonceng (distribusi normal), artinya error tidak condong pada salah satu sisi (under/over-predict) secara bias.
- **Heteroskedastisitas**: Jika titik tersebar acak di sekitar garis nol tanpa membentuk pola corong, berarti varians error stabil dan model layak digunakan di seluruh rentang daya beli.

## 8. Export Model Terbaik
Kita akan mengekspor pipeline model final (Ridge Regression) ke dalam format `.pkl` agar dapat dipanggil dalam produksi/aplikasi tanpa melatih ulang.

In [ ]:
# Buat folder models jika belum ada
os.makedirs('../models', exist_ok=True)

# Simpan model dengan joblib
model_filename = '../models/best_daya_beli_ridge.pkl'
joblib.dump(ridge_pipeline, model_filename)

print(f"Model berhasil disimpan di: {model_filename}")